# 09 — Results Aggregation

Loads every `results/*_results.json`, builds the consolidated comparison table
that goes into the thesis, and draws the subject-dependent vs subject-independent
bar chart that visualizes the generalization gap — the central finding of the work.

## 1. Load all result files

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
RESULTS_DIR = os.path.join(ROOT, "results")

from src.evaluation import load_all_results  # noqa: E402

blobs = load_all_results(RESULTS_DIR)
print(f"loaded {len(blobs)} result files:",
      ", ".join(sorted(b['model'] for b in blobs)))
LABEL_GUESS = next((b.get("config", {}).get("label") for b in blobs
                    if b.get("config", {}).get("label")), "valence")
print("target label:", LABEL_GUESS)

## 2. Flatten to one row per model

Each row: model, family, parameter count, and the accuracy / F1 (± std) under both
protocols, plus total training time.

In [ ]:
def _row(b):
    sd = b.get("subject_dependent", {}) or {}
    lo = b.get("loso", {}) or {}
    return {
        "model": b.get("model"),
        "family": b.get("family", ""),
        "params": (sd.get("params_count") or lo.get("params_count") or np.nan),
        "SD_acc": sd.get("accuracy", np.nan),
        "SD_f1": sd.get("f1", np.nan),
        "SD_acc_std": sd.get("accuracy_std", np.nan),
        "LOSO_acc": lo.get("accuracy", np.nan),
        "LOSO_f1": lo.get("f1", np.nan),
        "LOSO_acc_std": lo.get("accuracy_std", np.nan),
        "LOSO_f1_std": lo.get("f1_std", np.nan),
        "gap_acc": (sd.get("accuracy", np.nan) - lo.get("accuracy", np.nan)),
        "train_time_sec": (sd.get("train_time_sec", 0) or 0) + (lo.get("train_time_sec", 0) or 0),
    }

df = pd.DataFrame([_row(b) for b in blobs]).sort_values("LOSO_acc", ascending=False)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
df

## 3. Formatted comparison table (CSV + Markdown)

In [ ]:
def fmt_pm(m, s):
    if np.isnan(m):
        return "-"
    return f"{m:.3f} ± {s:.3f}" if not np.isnan(s) else f"{m:.3f}"

table = pd.DataFrame({
    "Model": df["model"],
    "Family": df["family"],
    "Params": df["params"].map(lambda p: f"{int(p):,}" if not np.isnan(p) else "-"),
    "SD Acc": [fmt_pm(m, s) for m, s in zip(df["SD_acc"], df["SD_acc_std"])],
    "SD F1": df["SD_f1"].map(lambda v: f"{v:.3f}" if not np.isnan(v) else "-"),
    "LOSO Acc": [fmt_pm(m, s) for m, s in zip(df["LOSO_acc"], df["LOSO_acc_std"])],
    "LOSO F1": [fmt_pm(m, s) for m, s in zip(df["LOSO_f1"], df["LOSO_f1_std"])],
    "SD-LOSO gap": df["gap_acc"].map(lambda v: f"{v:+.3f}" if not np.isnan(v) else "-"),
    "Train time (s)": df["train_time_sec"].map(lambda v: f"{v:,.0f}"),
})

csv_path = os.path.join(RESULTS_DIR, "benchmark_comparison.csv")
md_path = os.path.join(RESULTS_DIR, "benchmark_comparison.md")
table.to_csv(csv_path, index=False)
with open(md_path, "w") as f:
    f.write(table.to_markdown(index=False))
print("wrote", csv_path, "and", md_path)
print(table.to_markdown(index=False))

## 4. Generalization-gap bar chart

Subject-dependent vs subject-independent accuracy per model. The height difference
is the generalization gap the thesis is built around.

In [ ]:
d = df.dropna(subset=["SD_acc", "LOSO_acc"]).sort_values("SD_acc", ascending=False)
x = np.arange(len(d)); w = 0.38

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w/2, d["SD_acc"], w, yerr=d["SD_acc_std"].fillna(0),
            capsize=3, label="Subject-dependent")
b2 = ax.bar(x + w/2, d["LOSO_acc"], w, yerr=d["LOSO_acc_std"].fillna(0),
            capsize=3, label="Subject-independent (LOSO)")
ax.axhline(0.5, ls="--", lw=1, color="grey", label="chance")
ax.set_xticks(x); ax.set_xticklabels(d["model"], rotation=30, ha="right")
ax.set_ylabel("Accuracy"); ax.set_ylim(0.4, 1.0)
ax.set_title(f"DREAMER ({LABEL_GUESS}) — subject-dependent vs LOSO accuracy")
ax.legend()
for b in (b1, b2):
    ax.bar_label(b, fmt="%.2f", padding=2, fontsize=8)
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, "figures", "generalization_gap.png")
os.makedirs(os.path.dirname(fig_path), exist_ok=True)
plt.savefig(fig_path, dpi=150)
plt.show()
print("wrote", fig_path)

## 5. Accuracy vs efficiency

Parameter count (log x) against LOSO accuracy — the efficiency argument for the
lightweight models.

In [ ]:
e = df.dropna(subset=["params", "LOSO_acc"])
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(e["params"], e["LOSO_acc"])
for _, r in e.iterrows():
    ax.annotate(r["model"], (r["params"], r["LOSO_acc"]),
                textcoords="offset points", xytext=(5, 4), fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("parameters (log)"); ax.set_ylabel("LOSO accuracy")
ax.set_title("Accuracy vs efficiency (subject-independent)")
plt.tight_layout()
p = os.path.join(RESULTS_DIR, "figures", "accuracy_vs_efficiency.png")
plt.savefig(p, dpi=150); plt.show()
print("wrote", p)

## 6. Takeaways

* `benchmark_comparison.md` / `.csv` — paste-ready for the results chapter.
* `figures/generalization_gap.png` — the headline figure.
* `figures/accuracy_vs_efficiency.png` — the efficiency argument.
* Re-run any model notebook with a different `LABEL` and re-run this notebook to
  regenerate the whole set for arousal / dominance.